# Sentiment Analysis using LangDB and LLMs - Part 2

This is a  continuation to the post [here](https://langdb.ai/docs/getting_started/sentiment-1/). As discussed in the previous section, we will implement another approach in this example. 

**Approach**
- Send batch of 10 rows at a time leveraging `pretty_print()` function.
- Instruct LLM to lag out specific extreme reviews that deviate too much for a product in a response.
- Create an `Agent` for accessing reviews based on id.


In [2]:
-- Insert 100 records into products table
INSERT INTO products (product_id, product_name) 
SELECT number + 1, concat('Product ', toString(number + 1))
FROM numbers(100);

In [10]:
-- Insert randomised records into customer_reviews table with realistic review texts and anomalies
INSERT INTO customer_reviews (review_id, product_id, customer_id, review_date, review_text, rating, location)
SELECT 
    number + 1,
    rand() % 100 + 1,  -- Random product_id between 1 and 100
    rand() % 1000 + 1, -- Random customer_id between 1 and 1000
    today() - (rand() % 365), -- Random review_date within the last year
    [
        'This product exceeded my expectations. It is very reliable and works as described.',
        'I am not satisfied with this product. It broke within a week of purchase.',
        'Great value for money. I would recommend this to my friends and family.',
        'Terrible product. I had a very bad experience and would not buy again.',
        'The quality of this product is excellent. Very happy with my purchase.',
        'Not worth the price. There are better options available in the market.',
        'This product is just okay. It works but there are some issues.',
        'I love this product! It has made my life so much easier.',
        'The customer service was great, but the product did not meet my expectations.',
        'Good product but delivery was delayed.',
        -- Add anomalies
        'This product is amazing! (Anomaly: rating 1)',  -- Anomaly: Positive review with a very low rating
        'Worst product ever. (Anomaly: rating 5)',       -- Anomaly: Negative review with a very high rating
        'Totally different product received. (Anomaly: non-existent product_id)',  -- Anomaly: product_id > 100
        'Unbelievably good! Would buy again and again. (Anomaly: extremely high rating)',  -- Anomaly: rating 6
        'I will never buy this product again. (Anomaly: negative review with positive sentiment)',  -- Anomaly: Conflicting sentiment
        'The product arrived broken. (Anomaly: low rating but positive sentiment)'  -- Anomaly: Conflicting sentiment
    ][rand() % 15 + 1] AS review_text,  -- Select a random review text
    [1, 5, 4, 1, 5, 2, 3, 5, 3, 4, 1, 5, 1, 5, 6, 4, 2][rand() % 15 + 1] AS rating,  -- Introduce some anomalies in ratings
    [
        'New York, NY', 'Los Angeles, CA', 'Chicago, IL', 'Houston, TX', 'Phoenix, AZ', 
        'Philadelphia, PA', 'San Antonio, TX', 'San Diego, CA', 'Dallas, TX', 'San Jose, CA',
        'Miami, FL', 'Seattle, WA', 'Denver, CO', 'Boston, MA', 'Atlanta, GA',
        'Las Vegas, NV', 'Austin, TX', 'San Francisco, CA', 'Charlotte, NC', 'Detroit, MI'
    ][rand() % 15 + 1] AS location  -- Random location
FROM numbers(1000);

In [11]:
select count(*) from customer_reviews;

,count()
0,1000


In [12]:
select * from customer_reviews limit 10

,review_id,product_id,customer_id,review_date,review_text,rating,location
0,1,69,869,2023-07-06,"The customer service was great, but the product did not meet my expectations.",3,"Dallas, TX"
1,2,78,78,2023-06-02,Totally different product received. (Anomaly: non-existent product_id),1,"Denver, CO"
2,3,80,680,2024-01-21,Good product but delivery was delayed.,4,"San Jose, CA"
3,4,84,984,2024-05-01,Terrible product. I had a very bad experience and would not buy again.,1,"Houston, TX"
4,5,50,350,2023-11-12,The quality of this product is excellent. Very happy with my purchase.,5,"Phoenix, AZ"
5,6,19,319,2023-12-03,"The customer service was great, but the product did not meet my expectations.",3,"Dallas, TX"
6,7,80,80,2023-09-23,Good product but delivery was delayed.,4,"San Jose, CA"
7,8,47,847,2024-02-18,Worst product ever. (Anomaly: rating 5),5,"Seattle, WA"
8,9,58,458,2023-09-30,I love this product! It has made my life so much easier.,5,"San Diego, CA"
9,10,21,421,2023-09-12,Not worth the price. There are better options available in the market.,2,"Philadelphia, PA"


> Hint: You can use `pretty_print()` function to turn any table into YAML that can be fed into LLMs as context

In [17]:
select * from pretty_print((select * from customer_reviews limit 10))

,content
0,"| review_id | product_id | customer_id | review_date | review_text | rating | location |\n|-----------|------------|-------------|-------------|-------------------------------------------------------------------------------|--------|------------------|\n| 1 | 69 | 869 | 2023-07-06 | The customer service was great, but the product did not meet my expectations. | 3 | Dallas, TX |\n| 2 | 78 | 78 | 2023-06-02 | Totally different product received. (Anomaly: non-existent product_id) | 1 | Denver, CO |\n| 3 | 80 | 680 | 2024-01-21 | Good product but delivery was delayed. | 4 | San Jose, CA |\n| 4 | 84 | 984 | 2024-05-01 | Terrible product. I had a very bad experience and would not buy again. | 1 | Houston, TX |\n| 5 | 50 | 350 | 2023-11-12 | The quality of this product is excellent. Very happy with my purchase. | 5 | Phoenix, AZ |\n| 6 | 19 | 319 | 2023-12-03 | The customer service was great, but the product did not meet my expectations. | 3 | Dallas, TX |\n| 7 | 80 | 80 | 2023-09-23 | Good product but delivery was delayed. | 4 | San Jose, CA |\n| 8 | 47 | 847 | 2024-02-18 | Worst product ever. (Anomaly: rating 5) | 5 | Seattle, WA |\n| 9 | 58 | 458 | 2023-09-30 | I love this product! It has made my life so much easier. | 5 | San Diego, CA |\n| 10 | 21 | 421 | 2023-09-12 | Not worth the price. There are better options available in the market. | 2 | Philadelphia, PA |"


### Using Provider

We will be reusing the `open_ai_provider` we created in [`Using Providers`](Using%20Providers.ipynb).

If you haven't created one yet, create one with the following command:

In [ ]:
CREATE PROVIDER open_ai_provider
ENGINE = OpenAI(
	api_key='sk-proj-xxx',
	model_name = 'gpt-3.5-turbo'
);

## Creating Prompt and Model
Lets create a new model similar to review but that analyses 10 reviews at a time but also looks at exceptions and flag them.

In [33]:
CREATE PROMPT customer_review_pt_detailed (
  system 'You are an AI assistant. Look at the provided set of reviews in a markdown format and return average sentiment score ranging from 1 to 10 for each review in each line. 
    If you notice that provided rating is signifcantly different than calculated sentiment, mark anomaly as true. 
    Output should strictly be one single JSON array without any additional tags like below. Strictly return Response will be directly fed to a json parser. 
    ```
    [{ "review_id": review_id, "sentiment": sentiment, "anomaly": anomaly }]
    ```
    ',
  human '{{reviews}}'
);

In [34]:
CREATE MODEL review_detailed(
    reviews
) USING open_ai_provider()
PROMPT customer_review_pt_detailed;

In [40]:
CREATE TABLE calculated_review_jsons ( 
    batch_id UInt32,
    response String,
)
Engine = MergeTree
Order By response

### Batch & Scheduling
Lets run a batch request with a `batch_id` parameter. We can leverage `SPAWN TASK command` to run things in background or use external scheduling tools to execute these queries in a batch. 

In [42]:
INSERT INTO calculated_review_jsons
SELECT 1 as batch_id, review_detailed(
    (select content from pretty_print((select * from customer_reviews limit 10)) limit 1)
) 

In [43]:
INSERT INTO calculated_review_jsons
SELECT 2 as batch_id, review_detailed(
    (select content from pretty_print((select * from customer_reviews offset 10limit 10)) limit 1)
) 

In [45]:
select * from calculated_review_jsons limit 2

,batch_id,response
0,1,"[\n { ""review_id"": 1, ""sentiment"": 4, ""anomaly"": false },\n { ""review_id"": 2, ""sentiment"": 1, ""anomaly"": false },\n { ""review_id"": 3, ""sentiment"": 7, ""anomaly"": false },\n { ""review_id"": 4, ""sentiment"": 2, ""anomaly"": false },\n { ""review_id"": 5, ""sentiment"": 8, ""anomaly"": false },\n { ""review_id"": 6, ""sentiment"": 4, ""anomaly"": false },\n { ""review_id"": 7, ""sentiment"": 7, ""anomaly"": false },\n { ""review_id"": 8, ""sentiment"": 2, ""anomaly"": true },\n { ""review_id"": 9, ""sentiment"": 8, ""anomaly"": false },\n { ""review_id"": 10, ""sentiment"": 2, ""anomaly"": false }\n]"
1,2,"[\n {""review_id"": 11, ""sentiment"": 4, ""anomaly"": false},\n {""review_id"": 12, ""sentiment"": 2, ""anomaly"": true},\n {""review_id"": 13, ""sentiment"": 9, ""anomaly"": true},\n {""review_id"": 14, ""sentiment"": 2, ""anomaly"": true},\n {""review_id"": 15, ""sentiment"": 8, ""anomaly"": false},\n {""review_id"": 16, ""sentiment"": 8, ""anomaly"": true},\n {""review_id"": 17, ""sentiment"": 4, ""anomaly"": false},\n {""review_id"": 18, ""sentiment"": 3, ""anomaly"": false},\n {""review_id"": 19, ""sentiment"": 5, ""anomaly"": false},\n {""review_id"": 20, ""sentiment"": 9, ""anomaly"": true}\n]"


### Calculating avg sentiment based on LLM response

In [56]:
select cr.review_id, p.product_id, p.product_name, avg(sentiment)
FROM customer_reviews as cr 
JOIN products as p on p.product_id = cr.product_id
JOIN (
    SELECT
        JSONExtractInt(element, 'sentiment') AS sentiment,
        JSONExtractBool(element, 'anomaly') AS anomaly,
        JSONExtractInt(element, 'review_id') AS review_id
    FROM
    (
        SELECT batch_id, arrayJoin(JSONExtractArrayRaw(response)) AS element
        FROM calculated_review_jsons as crj 
    ) 
) crj on crj.review_id = cr.review_id
group by cr.review_id, p.product_id, p.product_name
limit 3

,cr.review_id,p.product_id,product_name,avg(sentiment)
0,10,21,Product 21,2
1,5,50,Product 50,8
2,17,2,Product B,4


Using the above query lets create an `agent` that returns avg sentiment based on a product id

In [65]:
CREATE AGENT product_sentiment (product_id Int) AS 
SELECT cr.review_id as review_id, p.product_id as product_id, p.product_name as product_name, avg(sentiment) as avg_sentiment
FROM customer_reviews as cr 
JOIN products as p on p.product_id = cr.product_id
JOIN (
    SELECT
        JSONExtractInt(element, 'sentiment') AS sentiment,
        JSONExtractBool(element, 'anomaly') AS anomaly,
        JSONExtractInt(element, 'review_id') AS review_id
    FROM
    (
        SELECT batch_id, arrayJoin(JSONExtractArrayRaw(response)) AS element
        FROM calculated_review_jsons as crj 
    ) 
) crj on crj.review_id = cr.review_id
WHERE p.product_id = $product_id
group by cr.review_id, p.product_id, p.product_name

In [67]:
select * from product_sentiment(21)

,review_id,product_id,product_name,avg_sentiment
0,10,21,Product 21,2


As this response is purely a SQL query, it can immediately integrate with your existing data or BI environments. 